In [172]:
# Library Import
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

path = "USvideos.csv"
# Dataset is now stored in a Pandas Dataframe

This Content-Based Recommendation System is based from the work of Saket Garodia (2020) with a few key differences:
* Instead of using a movie dataset, this implementation will be using a YouTube dataset of trending videos from the year 2019 by Mitchell J.
* 

Saket Garodia's article detailing the system can be found here: https://medium.com/analytics-vidhya/content-based-recommender-systems-in-python-2b330e01eb80

Mitchell J.'s dataset can be downloaded here: https://www.kaggle.com/datasnaek/youtube-new.

Our recommender algorithm will primarily focus on the four variables of a video contained below:
Title, Title of the Uploader, Category_ID, and the tags.

In [173]:
# Sample of 'videos' contents from rows 0 to 2.
fieldsToRead = ['title','channel_title','tags','category_id']
videos = pd.read_csv(path, skipinitialspace=True, usecols=fieldsToRead, nrows=5000)
videos.head(5)

,title,channel_title,category_id,tags
0,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,SHANtell martin
1,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,"last week tonight trump presidency|""last week ..."
2,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,"racist superman|""rudy""|""mancuso""|""king""|""bach""..."
3,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,"rhett and link|""gmm""|""good mythical morning""|""..."
4,I Dare You: GOING BALD!?,nigahiga,24,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""..."


In [174]:
videos['tags'] = videos['tags'].str.replace("|", " ")
videos['tags'] = videos['tags'].str.replace('"', "")
videos.head(5)

C:\Users\Jericho\AppData\Local\Temp\ipykernel_22548\876143360.py:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  videos['tags'] = videos['tags'].str.replace("|", " ")


,title,channel_title,category_id,tags
0,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,SHANtell martin
1,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,last week tonight trump presidency last week t...
2,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,racist superman rudy mancuso king bach racist ...
3,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,rhett and link gmm good mythical morning rhett...
4,I Dare You: GOING BALD!?,nigahiga,24,ryan higa higatv nigahiga i dare you idy rhpc ...


In [175]:
mapping = pd.Series(videos.index,index = videos["title"])
mapping

title
WE WANT TO TALK ABOUT OUR MARRIAGE                                                     0
The Trump Presidency: Last Week Tonight with John Oliver (HBO)                         1
Racist Superman | Rudy Mancuso, King Bach & Lele Pons                                  2
Nickelback Lyrics: Real or Fake?                                                       3
I Dare You: GOING BALD!?                                                               4
                                                                                    ... 
YOUTUBERS REACT TO THEIR OLD YOUTUBE CHANNEL PROFILE #2                             4995
Chris Stapleton - A Simple Song (Audio)                                             4996
Marvel Studios' Avengers: Infinity War Official Trailer                             4997
Prince Harry and Meghan Markle Get the Alison Hammond Experience! | This Morning    4998
Rolled Ice Cream DIY How to make rolled ice cream at home                           4999
Length: 5000, d

In [176]:
def recommend_based_on_tags(video_input):
    similarity_score = 0;
    
    video_input_row = videos.loc[videos['title'] == video_input]
    input_tags = videos['tags'].loc[videos['title'] == video_input]
    
    videos_to_recommend = mapping[video_input]
    similar_titles = set()
    similar_titles_indices = set()
    
    for index, row in videos.iterrows():
        input_tags_list = input_tags.iloc[0].split()
        index_row_tags = row['tags'].split()
        combined_list = set(input_tags_list)&set(index_row_tags)
        common_tags = sorted(combined_list, key = lambda x : input_tags_list.index(x))
        if len(common_tags) >= 3:
            similar_titles.add(videos.at[index, 'title'])
            similar_titles_indices.add(index)
    
    return videos['title'].loc[similar_titles_indices]

In [177]:
recommend_based_on_tags('Nickelback Lyrics: Real or Fake?')

C:\Users\Jericho\AppData\Local\Temp\ipykernel_22548\2225473872.py:20: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  return videos['title'].loc[similar_titles_indices]


4096    John Boyega Shows Off His Best Michael Jackson...
3                        Nickelback Lyrics: Real or Fake?
6               Roy Moore & Jeff Sessions Cold Open - SNL
2055          WHISPER CHALLENGE w/ MY MOM // Grace Helbig
4621                                          Getting Fit
                              ...                        
1524                       Diana Ross Lost Her Fanny Pack
502             Roy Moore & Jeff Sessions Cold Open - SNL
3572    Lesser Known BROADWAY Songs - Mystery Solo | T...
3573                                Alfred: Before Batman
4596        TEENS READ 10 FUNNY FRIEND ZONE TEXTS (REACT)
Name: title, Length: 229, dtype: object